<center>
    <img src="https://rockborne.com/wp-content/uploads/2021/07/LandingPage-Header-RED-CENTRE.jpg" width="900" alt="logo"  />
</center>

# Practical: Regression Model Development with Spark MLlib (Car Prices)

*Session 5 · Notebook 03.03 · Practical · Student version*

## About this notebook

This is the Session 4 regression capstone (notebook 02.06, car prices) **rebuilt in PySpark**. The data, the questions and the order are the same; only the tools change. You check data quality, explore, then build and compare a simple linear regression, a multiple linear regression, Ridge and Lasso, all scored on one shared test set.

Working through a problem you have already solved in scikit-learn is the fastest way to see what is genuinely different in Spark (feature vectors, pipelines of indexers and encoders, evaluators) and what is not (the modelling decisions). It also uses a different dataset from the lecture (notebook 03.02, California housing), and adds what the lecture left for the classification notebook: **categorical features**, indexed and one-hot encoded inside the pipeline.

## About the exercises

Each task appears as two cells:

1. A markdown cell describing what to do (sometimes with a `> *Hint:*`).
2. A `# Your turn` cell for you to write your answer.

This is the student version, so the answers are not included. Work top to bottom: later tasks reuse variables from earlier answers.

## About the data

The **automobile** dataset (201 cars from 1985 imports) describes each car's specifications and its **price**, the regression target.

| Column (after renaming) | Meaning |
|---|---|
| `price` | Price in dollars (target) |
| `engine_size`, `curb_weight`, `horsepower`, `width`, `highway_mpg` | Numeric specifications used as features |
| `drive_wheels` | fwd, rwd or 4wd |
| `body_style` | sedan, hatchback, wagon, hardtop, convertible |
| `aspiration` | std or turbo |

Original column names contain dashes (`engine-size`) and one contains a slash (`city-L/100km`); the setup renames them to `snake_case`, as in the walkthrough (notebook 01.02). 201 rows is tiny for Spark; the point is the workflow, which would be identical on 200 million rows.

## Index

- [Part 0: Setup](#p0)
- [Part 1: Data quality](#p1)
- [Part 2: Exploratory data analysis](#p2)
- [Modelling setup (shared split)](#msetup)
- [Part 3: Simple linear regression](#p3)
- [Part 4: Multiple linear regression](#p4)
- [Part 5: Regularisation (Ridge and Lasso)](#p5)
- [Part 6: Capstone: compare the models](#p6)
- [Challenge: everything in one pipeline](#challenge)
- [Further practice](#further)

<a id="p0"></a>
# Part 0: Setup

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))   # finds config.py next to this notebook or one level up

import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
SEED = 42
from config import pandas_bridge, session_datasets, session_datasets_http

# On Databricks this returns the session that already exists; on a laptop it starts one.
spark = SparkSession.builder.appName("05_03_03_Practical_CarPrices").getOrCreate()
print("Spark version:", spark.version)

In [ ]:
# Default: the pandas bridge. pandas downloads the file from the course S3 bucket over HTTPS and
# Spark converts the result. Works on every compute; fine for training-size files, never for
# real big data (the whole file passes through the driver's memory).
auto = pandas_bridge(spark, "automobile")

# Alternative 1: the same two steps in plain code. pandas reads the file, Spark converts it.
# (The bridge also turns pandas' missing values, NaN, into proper Spark nulls; this version does not.)
# pdf = pd.read_csv(session_datasets_http["automobile"])
# auto = spark.createDataFrame(pdf)

# Alternative 2: Spark reads the file straight from S3 (the big-data way; needs S3 access on your compute)
# auto = spark.read.csv(session_datasets["automobile"], header=True, inferSchema=True)

auto = auto.toDF(*[c.lower().replace("-", "_").replace("/", "_") for c in auto.columns])
print("rows:", auto.count(), "| columns:", len(auto.columns))
auto.select("make", "body_style", "drive_wheels", "engine_size", "horsepower", "price").show(5)

<a id="p1"></a>
# Part 1: Data quality

Before modelling, check the data is sound: missing values, duplicates and obvious outliers.

### Exercise 1.1: Missing values and duplicates

Count the nulls in each column and print only the columns that have any. Also print the number of exact duplicate rows.

In [ ]:
# Your turn

### Exercise 1.2: Outliers in the target

Using the IQR rule, count how many cars have an outlying `price` (above Q3 + 1.5 IQR or below Q1 - 1.5 IQR), and print the upper bound. Are they errors?

> *Hint:* `df.approxQuantile(col, [0.25, 0.75], 0.0)` returns the two quartiles as a Python list.

In [ ]:
# Your turn

### Exercise 1.3: Handle the missing values

Only a handful of rows have nulls. Drop them, assign the result back to `auto`, and confirm no nulls remain.

In [ ]:
# Your turn

<a id="p2"></a>
# Part 2: Exploratory data analysis

### Exercise 2.1: Distribution of the target

Print the summary statistics and the skewness of `price` (`F.skewness`), and plot a histogram (bring the single column to pandas). Is it symmetric?

In [ ]:
# Your turn

### Exercise 2.2: Which numeric features drive price?

Compute the correlation of every numeric column with `price` using `auto.stat.corr`, and show the top six by absolute value.

In [ ]:
# Your turn

### Exercise 2.3: Visualise the relationships

Draw a scatter of `engine_size` against `price`, and a boxplot of `price` by `drive_wheels`. The data is small, so bring the needed columns to pandas.

In [ ]:
# Your turn

<a id="msetup"></a>
# Modelling setup (shared split)

This part is done for you. Every model in Parts 4 and 5 uses the **same** features, the **same** train/test split and the **same** preparation steps, so the comparison at the end is fair. Three steps:

1. Choose the features and split the data.
2. Prepare the data once: four steps, each learned from `train` and applied to both parts, giving `train_prep` and `test_prep`.
3. A small helper, `record`, that scores a model's test predictions and keeps the result for the final table.

In [ ]:
# 1. The features: five numeric columns and three text (categorical) columns
numeric_features = ["engine_size", "curb_weight", "horsepower", "width", "highway_mpg"]
categorical_features = ["drive_wheels", "body_style", "aspiration"]

model_data = auto.select("price", *numeric_features, *categorical_features)
train, test = model_data.randomSplit([0.8, 0.2], seed=SEED)
print("train:", train.count(), "| test:", test.count())

**2. The preparation steps.** The same four steps as in the lectures, one per line. Text columns are turned into numbers first (index, then one-hot encode, as in lecture 03.04), then everything goes into one vector and is standardised (as in lecture 03.02).

In [ ]:
# a) give each category a number:            drive_wheels -> drive_wheels_idx
indexer = StringIndexer(inputCols=["drive_wheels", "body_style", "aspiration"],
                        outputCols=["drive_wheels_idx", "body_style_idx", "aspiration_idx"])

# b) one 0/1 column per category:             drive_wheels_idx -> drive_wheels_ohe
encoder = OneHotEncoder(inputCols=["drive_wheels_idx", "body_style_idx", "aspiration_idx"],
                        outputCols=["drive_wheels_ohe", "body_style_ohe", "aspiration_ohe"])

# c) pack the numeric and one-hot columns into one vector
assembler = VectorAssembler(inputCols=numeric_features + ["drive_wheels_ohe", "body_style_ohe", "aspiration_ohe"],
                            outputCol="features_raw")

# d) standardise, so the coefficients can be compared
scaler = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)

# Run the four steps. Each one is learned from the TRAINING data, then applied to both parts.
indexer_model = indexer.fit(train)
train_prep = indexer_model.transform(train)
test_prep = indexer_model.transform(test)

encoder_model = encoder.fit(train_prep)
train_prep = encoder_model.transform(train_prep)
test_prep = encoder_model.transform(test_prep)

train_prep = assembler.transform(train_prep)       # nothing to learn: it only packs the columns
test_prep = assembler.transform(test_prep)

scaler_model = scaler.fit(train_prep)
train_prep = scaler_model.transform(train_prep)
test_prep = scaler_model.transform(test_prep)

What the preparation did to one car: the text value, its number, its one-hot vector, and the final `features` vector the models read.

In [ ]:
train_prep.select("drive_wheels", "drive_wheels_idx", "drive_wheels_ohe", "features").show(3, truncate=60)

**3. The `record` helper.** It scores a model's predictions on the test set with the three metrics from the lecture and stores them, so Part 6 can compare every model in one table.

In [ ]:
r2_eval = RegressionEvaluator(labelCol="price", metricName="r2")
rmse_eval = RegressionEvaluator(labelCol="price", metricName="rmse")
mae_eval = RegressionEvaluator(labelCol="price", metricName="mae")

results = {}   # model name -> its scores

def record(name, pred, n_features):
    # pred = a model's predictions on the test set (the output of model.transform(...))
    r2, rmse, mae = r2_eval.evaluate(pred), rmse_eval.evaluate(pred), mae_eval.evaluate(pred)
    results[name] = {"R2": r2, "RMSE": rmse, "MAE": mae, "n_features": n_features}
    print(f"{name}: R2={r2:.3f}, RMSE={rmse:,.0f}, MAE={mae:,.0f}, features={n_features}")

<a id="p3"></a>
# Part 3: Simple linear regression

### Exercise 3.1: Fit and evaluate an SLR model

Even one feature must be in a vector. Use a `VectorAssembler` to put only `engine_size` into a column `engine_vec`, on both `train` and `test`. Fit a `LinearRegression(featuresCol="engine_vec", labelCol="price")` on the training part as `slr`, then record its test predictions with `record("SLR (engine_size)", slr.transform(...), n_features=1)`.

In [ ]:
# Your turn

### Exercise 3.2: Interpret it

Print the slope and intercept, and write the equation for price in terms of engine size.

In [ ]:
# Your turn

<a id="p4"></a>
# Part 4: Multiple linear regression

### Exercise 4.1: Fit and evaluate an MLR model

Fit a `LinearRegression(featuresCol="features", labelCol="price")` on `train_prep` as `mlr`, and record its predictions on `test_prep` as `"MLR"`. For `n_features`, use the number of coefficients (`len(mlr.coefficients)`).

> *Hint:* The data is already prepared: `fit` on `train_prep`, `transform` `test_prep`.

In [ ]:
# Your turn

### Exercise 4.2: Which features matter most?

The features were standardised, so coefficient sizes are comparable. Read the names from the `features_raw` column metadata (the helper is given) and list the five largest coefficients by absolute value.

In [ ]:
# Your turn

<a id="p5"></a>
# Part 5: Regularisation (Ridge and Lasso)

In scikit-learn you used `RidgeCV` and `LassoCV`, which choose the penalty strength by cross-validation. MLlib's equivalent, `CrossValidator`, cannot run on the course compute (lecture 03.02, Section 13), so here you first fit Ridge and Lasso with given strengths, then check those strengths yourself with a **validation split**. Ridge is `elasticNetParam=0.0`; Lasso is `elasticNetParam=1.0`; `regParam` is the strength (scikit-learn's `alpha`).

### Exercise 5.1: Ridge

Fit `LinearRegression(featuresCol="features", labelCol="price", regParam=100.0, elasticNetParam=0.0)` on `train_prep` as `ridge`, and record its predictions on `test_prep` as `"Ridge"`. For `n_features`, count the coefficients that are not zero.

> *Hint:* Price is in dollars, so useful penalty values are much larger than in the housing lecture.

In [ ]:
# Your turn

### Exercise 5.2: Lasso

Do the same for Lasso (`elasticNetParam=1.0`) with `regParam=300.0`. Record it as `"Lasso"`. How many features does it keep (non-zero coefficients) out of the total?

In [ ]:
# Your turn

### Exercise 5.3: Check the strengths with a validation split

Are 100 and 300 sensible strengths? Split `train_prep` again, 80/20 with `seed=SEED`, into `fit_part` and `val_part`. For Ridge and for Lasso, fit on `fit_part` with `regParam` in `[1, 10, 100, 300, 1000, 3000]` and print the RMSE on `val_part`. Which strength wins for each? (The test set stays untouched: it is only for the final comparison.)

> *Hint:* A validation split is a one-fold version of cross-validation: fit on one part, score on the other.

In [ ]:
# Your turn

<a id="p6"></a>
# Part 6: Capstone: compare the models

### Exercise 6.1: Build the comparison table

Turn `results` into a pandas DataFrame (models as rows), sort by R2 (best first) and display it rounded.

In [ ]:
# Your turn

### Exercise 6.2: Conclusions and recommendation

In a few sentences, recommend a model, referring to the table and weighing accuracy (R2, RMSE) against simplicity (number of features). How do the Spark results compare with your scikit-learn results from Session 4?

In [ ]:
# Your turn

<a id="challenge"></a>
# Challenge: everything in one pipeline

So far you prepared the data once, step by step, and fitted every model on the result. A `Pipeline` (lecture 03.02, Section 12) does the same in one object: raw data in, predictions out.

### Challenge: Build the Ridge pipeline

Build a `Pipeline` with five stages: the four preparation steps from the modelling setup (`indexer`, `encoder`, `assembler`, `scaler`) and the Ridge model of Exercise 5.1. Fit it on the **raw** `train`, score the raw `test`, and compare R2 and RMSE with `results["Ridge"]`. Why does a pipeline matter when the model is used on new data?

> *Hint:* Pass the raw DataFrames (`train`, `test`): the pipeline runs the preparation itself.

In [ ]:
# Your turn

<a id="further"></a>
## Further practice

- `price` is right-skewed: model `log(price)` (`F.log("price")`) and compare RMSE after transforming predictions back with `F.exp`.
- Add `make` as a categorical feature and see whether Lasso keeps its one-hot columns.
- Plot residuals (actual minus predicted) for the best model against its predictions.